# Análisis y figuras (local) — Pruebas 1–5

Notebook **local** que carga los `parquet` de todas las pruebas (bajados de Drive) y
genera las figuras. Desacoplado de Colab: iterás los gráficos sin re-correr nada.

- Métrica de fidelidad: **SI-SDR** (reemplaza al SDR clásico).
- `SIR` toma la columna ya corregida de los datos.
- Convención: cada prueba se busca por prefijo y se toma **el último run** (RUN_TAG).

Bajá la carpeta `results/` de Drive a `RESULTS_DIR` (ver Setup).

In [ ]:
import os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns; HAVE_SNS=True
except ImportError:
    HAVE_SNS=False; print("[!] seaborn no instalado (pip install seaborn) -> boxplots de P5 no corren")

# ===================== CONFIG =====================
RESULTS_DIR = "results"      # carpeta local con los subdirs P1_*/ P2_*/ ... bajados de Drive
FIG_DIR     = "figuras"      # donde se guardan los .png
os.makedirs(FIG_DIR, exist_ok=True)
# =================================================

def load_latest(prefix):
    """Carga el parquet del ultimo run cuyo subdir empieza con `prefix`."""
    cands = sorted(glob.glob(os.path.join(RESULTS_DIR, prefix+"*")))
    if not cands:
        print(f"[!] no encontrado: {prefix}*  en {RESULTS_DIR}")
        return None
    pq = os.path.join(cands[-1], "mird_benchmark_metrics.parquet")
    if not os.path.isfile(pq):
        pq = os.path.join(cands[-1], "mird_benchmark_metrics.csv")
        if not os.path.isfile(pq): print(f"[!] sin metrics en {cands[-1]}"); return None
        return pd.read_csv(pq)
    print(f"[*] {prefix}: {os.path.basename(cands[-1])}")
    return pd.read_parquet(pq)

def savefig(fig, name):
    p = os.path.join(FIG_DIR, name); fig.savefig(p, dpi=140, bbox_inches="tight")
    print("  ->", p)

# Metricas (SI-SDR en vez de SDR). Δ end-to-end vs early.
DTOT = {"PESQ":"Delta_tot_PESQ_early","STOI":"Delta_tot_STOI_early",
        "SI-SDR":"Delta_tot_SI-SDR_early","SIR":"Delta_tot_SIR_early",
        "SAR":"Delta_tot_SAR_early","CD":"Delta_tot_CD_early"}

## Prueba 1 — Robustez (DS / MVDR-geo / NM-MVDR)

In [ ]:
# 3 sub-corridas: 2a (DOA), 2b_gain, 2b_phase. Curvas de Δ vs la variable barrida.
PROCS=["DS","MVDR-geo","NM-MVDR"]; COL={"DS":"tab:green","MVDR-geo":"tab:red","NM-MVDR":"tab:orange"}
MET1=[("PESQ",DTOT["PESQ"]),("STOI",DTOT["STOI"]),("SI-SDR",DTOT["SI-SDR"]),("SIR",DTOT["SIR"])]

def curvas(df, xcol, xlabel, title, fname):
    if df is None: return
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for ax,(lbl,col) in zip(axes.ravel(),MET1):
        if col not in df.columns: ax.set_title(f"(falta {col})"); continue
        for pr in PROCS:
            sub=df[df.processor==pr]
            if sub.empty: continue
            g=sub.groupby(xcol)[col]; m,s=g.mean(),g.std()
            ax.plot(m.index,m.values,"-o",color=COL.get(pr,"gray"),ms=5,label=pr)
            ax.fill_between(m.index,(m-s).values,(m+s).values,color=COL.get(pr,"gray"),alpha=0.12)
        ax.set_xlabel(xlabel); ax.set_ylabel(f"Δ {lbl}"); ax.grid(alpha=0.3)
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout(); savefig(fig,fname); plt.show()

curvas(load_latest("P1_robustez_2a_doa"),  "error_angle_deg","error DOA [°]",
       "P1a — Δ vs error de DOA","P1_doa.png")
curvas(load_latest("P1_robustez_2b_gain"), "mismatch_gain","desajuste ganancia [dB]",
       "P1b — Δ vs mismatch de ganancia","P1_gain.png")
curvas(load_latest("P1_robustez_2b_phase"),"mismatch_phase","desajuste fase [°]",
       "P1b — Δ vs mismatch de fase","P1_phase.png")

## Prueba 2 — Calibración WPE (taps × delay × RT)

In [ ]:
df=load_latest("P2_calibracion_wpe")
if df is not None:
    df=df[df.processor=="NM-MVDR"].copy()
    def piv(m): return df.groupby(["wpe_taps","wpe_delay"])[m].mean().unstack("wpe_delay")
    print("SUPERFICIE taps(filas) x delay(cols) | media sobre RT+escenas")
    for lbl,m in [("PESQ",DTOT["PESQ"]),("SI-SDR",DTOT["SI-SDR"]),("CD",DTOT["CD"])]:
        if m in df.columns: print(f"\n--- {lbl} ---\n", piv(m).round(3).to_string())
    m5=[DTOT[k] for k in ["PESQ","STOI","SI-SDR","SIR","CD"] if DTOT[k] in df.columns]
    print("\n=== taps=5 | metricas por delay (elegir delay*) ===")
    print(df[df.wpe_taps==5].groupby("wpe_delay")[m5].mean().round(3).to_string())
    DELAY_SHOW=1
    mt=[DTOT[k] for k in ["PESQ","SI-SDR","CD"] if DTOT[k] in df.columns]
    mt+=[c for c in ["Delta_wpe_PESQ_early","Delta_wpe_SI-SDR_early"] if c in df.columns]
    print(f"\n=== delay={DELAY_SHOW} | sensibilidad a taps (honestidad HW) ===")
    print(df[df.wpe_delay==DELAY_SHOW].groupby("wpe_taps")[mt].mean().round(3).to_string())

## Prueba 3 — Efecto de WPE (on/off)

In [ ]:
df=load_latest("P3_efecto_wpe")
PLOT3=[("PESQ",DTOT["PESQ"]),("STOI",DTOT["STOI"]),("SI-SDR",DTOT["SI-SDR"]),("SIR",DTOT["SIR"])]
procs3=["DS","NM-MVDR","ORACLE-SCM"]; c3={"DS":"tab:green","NM-MVDR":"tab:orange","ORACLE-SCM":"tab:blue"}
if df is not None:
    fig,axes=plt.subplots(2,2,figsize=(12,8))
    for ax,(lbl,col) in zip(axes.ravel(),PLOT3):
        if col not in df.columns: continue
        for pr in procs3:
            for wpe,ls,mk in [(True,"-","o"),(False,"--","x")]:
                sub=df[(df.processor==pr)&(df.use_wpe==wpe)]
                if sub.empty: continue
                g=sub.groupby("rt60")[col]; m=g.mean()
                ax.plot(m.index*1000,m.values,ls,color=c3.get(pr,"gray"),marker=mk,ms=5,
                        label=f"{pr} {'WPE' if wpe else 'noWPE'}")
        ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(f"Δ {lbl}"); ax.grid(alpha=0.3)
    axes.ravel()[0].legend(fontsize=7,ncol=3)
    fig.suptitle("P3 — Δ vs RT60 | WPE on (—) vs off (--)"); fig.tight_layout()
    savefig(fig,"P3_wpe_onoff.png"); plt.show()

## Prueba 4 — Selección de post-filtro (trade-off)

In [ ]:
df=load_latest("P4_mono_postfiltro")
if df is not None:
    cols=[DTOT[k] for k in ["PESQ","STOI","SI-SDR","SIR","SAR"] if DTOT[k] in df.columns]
    agg=df.groupby("processor")[cols].mean()
    print("=== Δ por procesador (trade-off) ===\n", agg.round(3).sort_values(DTOT["PESQ"],ascending=False).to_string())
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for ax,(xc,xl) in zip(axes,[(DTOT["STOI"],"Δ STOI"),(DTOT["SAR"],"Δ SAR")]):
        if xc not in agg.columns: continue
        for name,r in agg.iterrows():
            fam="BANPF" if name.startswith("BANPF") else ("PF" if name.startswith("PF_") else "otro")
            ax.scatter(r[xc],r[DTOT["PESQ"]],c={"BANPF":"tab:blue","PF":"tab:red","otro":"gray"}[fam],s=60)
            ax.annotate(name,(r[xc],r[DTOT["PESQ"]]),fontsize=7,xytext=(4,4),textcoords="offset points")
        ax.set_xlabel(xl); ax.set_ylabel("Δ PESQ"); ax.grid(alpha=0.3)
    fig.suptitle("P4 — trade-off del post-filtro (elegir smooth*)"); fig.tight_layout()
    savefig(fig,"P4_tradeoff.png"); plt.show()

## Prueba 5 — Sistema vs mono (boxplots marginales)

In [ ]:
df=load_latest("P5_sys_vs_mono")
SYSTEM="Sistema"; METB=["SI-SDR","SIR","STOI","PESQ"]
PAL={"NM-MVDR":"tab:orange","Sistema":"tab:blue","DTLN-mono":"tab:gray"}
if df is not None and HAVE_SNS:
    def tidy(df):
        rows=[]
        for method,proc in [("NM-MVDR","NM-MVDR"),("Sistema",SYSTEM)]:
            for _,r in df[df.processor==proc].iterrows():
                for M in METB:
                    rows.append({"method":method,"metric":M,"value":r.get(f"proc_{M}_early",np.nan),
                                 "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
        for _,r in df[df.processor=="NM-MVDR"].iterrows():
            for M in METB:
                rows.append({"method":"DTLN-mono","metric":M,"value":r.get(f"dtln_alone_{M}_early",np.nan),
                             "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
        return pd.DataFrame(rows).dropna(subset=["value"])
    T=tidy(df); order=["NM-MVDR","Sistema","DTLN-mono"]
    def boxfig(xv,xl,title,fn):
        fig,axes=plt.subplots(2,2,figsize=(14,9))
        for ax,M in zip(axes.ravel(),METB):
            sns.boxplot(data=T[T.metric==M],x=xv,y="value",hue="method",hue_order=order,
                        palette=PAL,ax=ax,fliersize=1,linewidth=0.8)
            ax.set_xlabel(xl); ax.set_ylabel(M); ax.grid(alpha=0.3,axis="y")
            if ax.get_legend(): ax.get_legend().remove()
        axes.ravel()[0].legend(fontsize=8)
        fig.suptitle(title); fig.tight_layout(); savefig(fig,fn); plt.show()
    boxfig("iSIR","iSIR [dB]","P5 — vs iSIR","P5_isir.png")
    boxfig("RT","RT60 [ms]","P5 — vs RT60","P5_rt.png")
    boxfig("N","nº interferentes","P5 — vs nº interferentes","P5_count.png")

    # Diferencia pareada (BF - mono)
    def tidy_d(df):
        rows=[]
        for cmp,proc in [("NM-MVDR−mono","NM-MVDR"),("Sistema−mono",SYSTEM)]:
            for _,r in df[df.processor==proc].iterrows():
                for M in METB:
                    rows.append({"cmp":cmp,"metric":M,
                                 "diff":r.get(f"proc_{M}_early",np.nan)-r.get(f"dtln_alone_{M}_early",np.nan),
                                 "iSIR":r["isir_db"],"RT":int(r["rt60"]*1000),"N":int(r["N_interferences"])})
        return pd.DataFrame(rows).dropna(subset=["diff"])
    D=tidy_d(df); PALD={"NM-MVDR−mono":"tab:orange","Sistema−mono":"tab:blue"}
    def diffig(xv,xl,title,fn):
        fig,axes=plt.subplots(2,2,figsize=(14,9))
        for ax,M in zip(axes.ravel(),METB):
            sns.boxplot(data=D[D.metric==M],x=xv,y="diff",hue="cmp",
                        hue_order=["NM-MVDR−mono","Sistema−mono"],palette=PALD,ax=ax,fliersize=1,linewidth=0.8)
            ax.axhline(0,color="k",lw=0.8,ls="--")
            ax.set_xlabel(xl); ax.set_ylabel(f"Δ {M} (BF−mono)"); ax.grid(alpha=0.3,axis="y")
            if ax.get_legend(): ax.get_legend().remove()
        axes.ravel()[0].legend(fontsize=8)
        fig.suptitle(title); fig.tight_layout(); savefig(fig,fn); plt.show()
    diffig("iSIR","iSIR [dB]","P5 — ventaja sobre mono vs iSIR","P5_diff_isir.png")
    diffig("N","nº interferentes","P5 — ventaja sobre mono vs nº interf","P5_diff_count.png")